In [1]:
import pandas as pd
import numpy as np
from gurobipy import Model, GRB, quicksum
from scipy.spatial.distance import cdist
import folium
import re

# CONSTANTS
bnb_mapping = {
    'ams': (52.37727, 4.83925),
    'rot': (51.9225, 4.4792),
    'utr': (52.0928, 5.1045),
    'hrl': (52.3800, 4.6400)
}
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred',
          'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white',
          'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# Load dataset
df_master = pd.read_csv("final attractions.csv")

# Mapping of city input to prefix
city_mapping = {
    'amsterdam': 'ams',
    'utrecht': 'utr',
    'rotterdam': 'rot',
    'haarlem': 'hrl',
    'the hague': 'hag',
    'eindhoven': 'ein',
}

def plan_itinerary(df_slice, ab_coords):
    num_days = int(input("How many days will you be staying? "))
    D = list(range(num_days))

    print("\nAvailable attractions:")
    print(df_slice['Attraction Name'].tolist())
    priority_names = []
    while True:
        user_input = input("Enter a priority attraction name (or type 'done' to finish): ").strip()
        if user_input.lower() == 'done':
            break
        elif user_input in df_slice['Attraction Name'].values:
            priority_names.append(user_input)
        else:
            print("Attraction not found. Please try again.")

    df = df_slice.copy().dropna(subset=["Latitude", "Longitude", "Opening Hours"]).reset_index(drop=True)
    I = df.index.tolist()
    n_attractions = len(I)

    df['Priority'] = df['Attraction Name'].apply(lambda x: 1 if x in priority_names else 0)
    df['Satisfaction Weight'] = df.apply(
        lambda row: 5 if row['Priority'] == 1 else row['Satisfaction Score (X_i)'],
        axis=1
    )
    X_i = pd.Series(df['Satisfaction Weight'].values, index=df.index)

    # Parse opening hours
    open_times, close_times = [], []
    for raw in df["Opening Hours"]:
        match = re.findall(r"(\d{1,2})(am|pm)", raw)
        if match and len(match) >= 2:
            start_h = int(match[0][0]) % 12 + (12 if match[0][1] == "pm" else 0)
            end_h = int(match[1][0]) % 12 + (12 if match[1][1] == "pm" else 0)
            open_times.append(start_h * 60)
            close_times.append(end_h * 60)
        else:
            open_times.append(0)
            close_times.append(1440)
    df["Open_min"] = open_times
    df["Close_min"] = close_times

    coords = df[["Latitude", "Longitude"]].to_numpy()
    D_matrix = cdist(coords, coords, metric='euclidean') * 111

    T_i = pd.Series(180, index=I)
    T_max = 480
    n = df['Priority'].sum() if df['Priority'].sum() > 0 else len(df)
    E_id = {(i, d): int(df.loc[i, "Close_min"] - df.loc[i, "Open_min"] >= T_i[i]) for i in I for d in D}

    model = Model("LOP")
    model.setParam("OutputFlag", 0)
    Y = model.addVars(I, D, vtype=GRB.BINARY, name="Y")
    U = model.addVars(I, D, vtype=GRB.INTEGER, name="U")
    F = model.addVars(I, I, D, vtype=GRB.BINARY, name="F")
    model.setObjective((5 / n) * quicksum(X_i[i] * Y[i, d] for i in I for d in D), GRB.MAXIMIZE)

    for d in D:
        model.addConstr(
            quicksum(T_i[i] * Y[i, d] for i in I) +
            quicksum(D_matrix[i][j] * F[i, j, d] for i in I for j in I) <= T_max
        )
    for i in I:
        for d in D:
            model.addConstr(Y[i, d] <= E_id[i, d])
            model.addConstr(quicksum(Y[i, d] for d in D) <= 1)
    for i in I:
        for d in D:
            model.addConstr(
                quicksum(F[i, j, d] for j in I) -
                quicksum(F[j, i, d] for j in I) == 0
            )
    M = n_attractions
    for d in D:
        for i in I:
            for j in I:
                model.addConstr(U[j, d] >= U[i, d] + 1 - M * (1 - F[i, j, d]))

    model.optimize()
    print(f"\nOverall Satisfaction Score Achieved: {model.objVal:.2f}")


    # Build schedule
    schedule = []
    for d in D:
        day_attractions = sorted([i for i in I if Y[i, d].X > 0.5], key=lambda x: (-X_i[x], U[x, d].X))
        if day_attractions:
            first_idx = day_attractions[0]
            first_open = df.loc[first_idx, 'Open_min']
            first_coords = df.loc[first_idx, ['Latitude', 'Longitude']].to_numpy().reshape(1, -1).astype(float)
            ab_travel_min = int(cdist([ab_coords], first_coords, metric='euclidean')[0][0] * 111 / 30 * 60)
            ab_depart_min = first_open - ab_travel_min
            current_time = first_open

            ab_dep_time = f"{int(ab_depart_min // 60):02d}:{int(ab_depart_min % 60):02d}"
            ab_arr_time = f"{int(first_open // 60):02d}:{int(first_open % 60):02d}"
            schedule.append([f"Day {d+1}", ab_dep_time, ab_arr_time, "Depart from Airbnb", f"{ab_travel_min} mins", "-"])

        for idx, i in enumerate(day_attractions):
            if current_time < df.loc[i, 'Open_min']:
                current_time = df.loc[i, 'Open_min']
            if 720 <= current_time < 810: current_time = 810
            if 1080 <= current_time < 1170: current_time = 1170

            start = current_time
            end = current_time + T_i[i]
            schedule.append([
                f"Day {d+1}",
                f"{int(start // 60):02d}:{int(start % 60):02d}",
                f"{int(end // 60):02d}:{int(end % 60):02d}",
                df.loc[i, 'Attraction Name'],
                f"{int(T_i[i] // 60)} hrs",
                X_i[i]
            ])
            current_time = end

            if idx + 1 < len(day_attractions):
                j = day_attractions[idx + 1]
                dist_km = D_matrix[i][j]
                travel_time = int((dist_km / 30) * 60)
                current_time += travel_time

    schedule_df = pd.DataFrame(schedule, columns=["Day", "Start Time", "End Time", "Attraction", "Duration", "Priority Score"])
    print("\nPlanned Itinerary:")
    print(schedule_df.to_string(index=False))

    # Map
    maps = []
    for d in D:
        color = colors[d % len(colors)]
        m_day = folium.Map(location=[df['Latitude'].mean(), df['Longitude'].mean()], zoom_start=13)
        path_coords = []
        day_indices = sorted([i for i in I if Y[i, d].X > 0.5], key=lambda x: (-X_i[x], U[x, d].X))

        folium.Marker(location=ab_coords, popup=f"Airbnb (Start/End) Day {d+1}", icon=folium.Icon(color='black', icon='home')).add_to(m_day)
        path_coords.append(ab_coords)

        for i in day_indices:
            row = df.loc[i]
            path_coords.append((row['Latitude'], row['Longitude']))
            folium.Marker(location=[row['Latitude'], row['Longitude']], popup=f"{row['Attraction Name']} (Day {d+1})", icon=folium.Icon(color=color)).add_to(m_day)

        path_coords.append(ab_coords)
        folium.Marker(location=ab_coords, popup=f"Return to Airbnb (Day {d+1})", icon=folium.Icon(color='gray', icon='home')).add_to(m_day)
        if path_coords:
            folium.PolyLine(locations=path_coords, color=color, weight=2.5).add_to(m_day)

        maps.append((f"Day {d+1} Map", m_day))

    for title, m_day in maps:
        print(title)
        display(m_day)

# MAIN LOOP
while True:
    city_input = input("\nWhich city are you interested in? (e.g., Amsterdam, Utrecht, Rotterdam, Haarlem): ").strip().lower()
    if city_input not in city_mapping:
        print("City not recognized. Try again.")
        continue

    city_prefix = city_mapping[city_input]
    ab_coords = bnb_mapping.get(city_prefix, (52.37727, 4.83925))  # fallback to Amsterdam
    df_slice = df_master[df_master['Location ID'].str.startswith(city_prefix)].copy()

    if df_slice.empty:
        print(f"No attractions found for {city_input.title()}. Please try another city.")
        continue

    plan_itinerary(df_slice, ab_coords)

    another = input("\nDo you want to plan another itinerary? (yes/no): ").strip().lower()
    if another != 'yes':
        print("Thank you for using the itinerary planner! Please give us A for the project :)")
        break



Which city are you interested in? (e.g., Amsterdam, Utrecht, Rotterdam, Haarlem):  Amsterdam
How many days will you be staying?  5



Available attractions:
['Rijksmuseum', 'Van Gogh Museum', 'Anne Frank House', 'Vondelpark', 'Dam Square', 'Royal Palace Amsterdam', 'Heineken Experience', 'Rembrandt House Museum', 'NEMO Science Museum', 'ARTIS Zoo', 'A’DAM Lookout', 'Amsterdam Museum', 'Bloemenmarkt', 'Moco Museum', 'Eye Filmmuseum']


Enter a priority attraction name (or type 'done' to finish):  Dam Square
Enter a priority attraction name (or type 'done' to finish):  done


Set parameter Username
Set parameter LicenseID to value 2611986
Academic license - for non-commercial use only - expires 2026-01-20

Overall Satisfaction Score Achieved: 165.00

Planned Itinerary:
  Day Start Time End Time             Attraction Duration Priority Score
Day 1      09:51    10:00     Depart from Airbnb   9 mins              -
Day 1      10:00    13:00       Anne Frank House    3 hrs              5
Day 1      13:30    16:30            Moco Museum    3 hrs              3
Day 2      09:47    10:00     Depart from Airbnb  13 mins              -
Day 2      10:00    13:00          A’DAM Lookout    3 hrs              3
Day 2      13:30    16:30    Heineken Experience    3 hrs              2
Day 3      08:50    09:00     Depart from Airbnb  10 mins              -
Day 3      09:00    12:00        Van Gogh Museum    3 hrs              4
Day 3      13:30    16:30       Amsterdam Museum    3 hrs              3
Day 4      08:49    09:00     Depart from Airbnb  11 mins              -


Day 2 Map


...

Day 3 Map


Day 4 Map


Day 5 Map



Do you want to plan another itinerary? (yes/no):  no


Thank you for using the itinerary planner! Please give us A for the project :)
